# 캡스톤디자인 2 — 모델링
**에네레기파 (임단군, 정예민, 안유경) | 수원대학교 데이터과학부**

**과업:** 불법 주정차 대응 정책 수립을 위한 주차 단속정보 분석  
**목표:** 단속건수 시계열 예측 → *어디서 × 언제* 단속을 집중해야 하는지 정책 근거 제공

### 모델 구성
| 단계 | 내용 |
|------|------|
| ① 시계열 예측 | Ridge / SVR / LightGBM / SARIMA / Prophet 비교 |
| ② 교차 분석  | 공간 위험도 × 시계열 패턴 → 단속 우선순위 도출 |

## 1. 라이브러리 임포트

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

from sklearn.linear_model import Ridge
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import lightgbm as lgb
from statsmodels.tsa.statespace.sarimax import SARIMAX
from prophet import Prophet

print("라이브러리 로드 완료")

## 2. 데이터 로드

In [ ]:
v2   = pd.read_csv("../data/전처리_기존/월별통합데이터_전처리_v2.csv", encoding="utf-8-sig")
risk = pd.read_csv("../data/전처리_기존/행정동별위험도점수.csv",        encoding="utf-8-sig")

print("월별통합 Shape:", v2.shape)
print("컬럼:", v2.columns.tolist())
display(v2.tail())
print()
print("행정동별 위험도 Shape:", risk.shape)
display(risk[["행정동명","위험도점수","위험등급"]].head())

## 3. 피처 엔지니어링

- `총상가수` 제거 — 전 기간 고정값(52,635), 분별력 없음  
- `계절` 원-핫 인코딩  
- `월` 사이클 인코딩 — 12월→1월 연속성 표현 (sin/cos)

In [ ]:
df = v2.copy()

# 총상가수 제거
df = df.drop(columns=["총상가수"])

# 계절 원-핫 인코딩
df = pd.get_dummies(df, columns=["계절"], prefix="계절")

# 월 사이클 인코딩
df["월_sin"] = np.sin(2 * np.pi * df["월"] / 12)
df["월_cos"] = np.cos(2 * np.pi * df["월"] / 12)

# 날짜 컬럼 (시각화·분할용)
df["날짜"] = pd.to_datetime(df[["연도","월"]].assign(day=1))
df = df.sort_values("날짜").reset_index(drop=True)

feature_cols = [c for c in df.columns if c not in ["단속건수","날짜","연도","월"]]
print("사용 피처:", feature_cols)
display(df.head())

## 4. 학습 / 테스트 분할

> **시계열 데이터는 랜덤 분할 금지** — 미래 데이터가 학습에 섞이면 leakage 발생

- Train : 2021.02 ~ 2024.12 (47행)  
- Test  : 2025.01 ~ 2025.04 (4행)

In [ ]:
train = df[df["날짜"] <= "2024-12-01"].copy()
test  = df[df["날짜"] >= "2025-01-01"].copy()

X_train = train[feature_cols]
y_train = train["단속건수"]
X_test  = test[feature_cols]
y_test  = test["단속건수"]

print(f"Train: {len(train)}행  ({train['날짜'].min().strftime('%Y.%m')} ~ {train['날짜'].max().strftime('%Y.%m')})")
print(f"Test : {len(test)}행  ({test['날짜'].min().strftime('%Y.%m')} ~ {test['날짜'].max().strftime('%Y.%m')})")
print(f"피처 수: {len(feature_cols)}개")

def evaluate(name, y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    print(f"{name:<12} MAE: {mae:>6.0f}건   RMSE: {rmse:>6.0f}건")
    return mae, rmse

results = {}

## 5-1. Ridge Regression (베이스라인)

In [ ]:
scaler     = StandardScaler()
X_tr_sc    = scaler.fit_transform(X_train)
X_te_sc    = scaler.transform(X_test)

ridge      = Ridge(alpha=1.0)
ridge.fit(X_tr_sc, y_train)
pred_ridge = ridge.predict(X_te_sc)

mae, rmse = evaluate("Ridge", y_test, pred_ridge)
results["Ridge"] = {"MAE": mae, "RMSE": rmse, "pred": pred_ridge}

coef_df = pd.DataFrame({"feature": feature_cols, "coef": ridge.coef_}).sort_values("coef", key=abs, ascending=False)
display(coef_df.head(6))

## 5-2. SVR (Support Vector Regression)

In [ ]:
svr      = SVR(kernel="rbf", C=5000, epsilon=500)
svr.fit(X_tr_sc, y_train)
pred_svr = svr.predict(X_te_sc)

mae, rmse = evaluate("SVR", y_test, pred_svr)
results["SVR"] = {"MAE": mae, "RMSE": rmse, "pred": pred_svr}

## 5-3. LightGBM

In [ ]:
lgbm_model = lgb.LGBMRegressor(
    n_estimators=300, learning_rate=0.03,
    num_leaves=15, min_child_samples=5, random_state=42
)
lgbm_model.fit(X_train, y_train)
pred_lgbm = lgbm_model.predict(X_test)

mae, rmse = evaluate("LightGBM", y_test, pred_lgbm)
results["LightGBM"] = {"MAE": mae, "RMSE": rmse, "pred": pred_lgbm}

fi = pd.DataFrame({"feature": feature_cols, "importance": lgbm_model.feature_importances_})
fi = fi.sort_values("importance", ascending=False)
print("\n피처 중요도 (상위 6):")
display(fi.head(6))

## 5-4. SARIMA

- order=(1,1,1) · seasonal_order=(1,0,1,12)  
- 외생변수 없이 순수 시계열 패턴만 모델링  
- **D=0** (계절 차분 생략): 51행 단기 시계열에서 계절 차분 시 데이터 소실 과다

In [ ]:
train_ts = train.set_index("날짜")["단속건수"]

sarima_model = SARIMAX(train_ts, order=(1,1,1), seasonal_order=(1,0,1,12))
sarima_fit   = sarima_model.fit(disp=False)
pred_sarima  = sarima_fit.forecast(steps=len(test)).values

mae, rmse = evaluate("SARIMA", y_test, pred_sarima)
results["SARIMA"] = {"MAE": mae, "RMSE": rmse, "pred": pred_sarima}

print("\nSARIMA AIC:", round(sarima_fit.aic, 2))

## 5-5. Prophet

In [ ]:
train_p = train[["날짜","단속건수"]].rename(columns={"날짜":"ds","단속건수":"y"})

m = Prophet(
    seasonality_mode="additive",
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False
)
m.fit(train_p)

future   = m.make_future_dataframe(periods=len(test), freq="MS")
forecast = m.predict(future)
pred_prophet = forecast.tail(len(test))["yhat"].values

mae, rmse = evaluate("Prophet", y_test, pred_prophet)
results["Prophet"] = {"MAE": mae, "RMSE": rmse, "pred": pred_prophet}

fig_comp = m.plot_components(forecast)
plt.tight_layout()
plt.show()

## 6. 모델 성능 비교

In [ ]:
comp_df = pd.DataFrame(
    {k: {"MAE": round(v["MAE"]), "RMSE": round(v["RMSE"])} for k, v in results.items()}
).T.sort_values("MAE")

print("=== 모델 성능 비교 (MAE 오름차순) ===")
display(comp_df)
best_name = comp_df.index[0]
print(f"\n최적 모델: {best_name}  MAE={comp_df.loc[best_name, 'MAE']:.0f}건")

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()
test_dates = test["날짜"].values

for i, (mname, v) in enumerate(results.items()):
    ax = axes[i]
    ax.plot(train["날짜"], y_train, color="steelblue", label="학습", lw=1.5)
    ax.plot(test_dates, y_test.values, color="black", label="실제", marker="o", lw=2)
    ax.plot(test_dates, v["pred"], color="crimson", label="예측", marker="s", ls="--", lw=2)
    ax.set_title(f"{mname}  MAE={v['MAE']:.0f}건")
    ax.legend(fontsize=8)
    ax.tick_params(axis="x", rotation=30)

ax = axes[-1]
comp_df["MAE"].plot(kind="bar", ax=ax, color="steelblue", edgecolor="black")
ax.set_title("모델별 MAE 비교")
ax.set_ylabel("MAE (건)")
ax.tick_params(axis="x", rotation=30)

plt.suptitle("단속건수 예측 — 모델 성능 비교", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig("../outputs/modeling_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 6-1. Prophet + SARIMA 가중 앙상블

단일 모델 대신 **MAE 역수 가중 앙상블**로 최종 예측값 도출.

**앙상블 가중치 계산:**
- w_Prophet = (1/MAE_Prophet) / (1/MAE_Prophet + 1/MAE_SARIMA)
- w_SARIMA  = (1/MAE_SARIMA) / (1/MAE_Prophet + 1/MAE_SARIMA)

검증 구간(2025.01~04) MAE 기반으로 더 정확한 모델에 더 높은 가중치 부여.  
대시보드(`dashboard/app.py`)에서 향후 6개월 예측에 이 앙상블을 사용.

In [ ]:
# MAE 역수 가중 앙상블 (Prophet + SARIMA)
mae_prophet = results["Prophet"]["MAE"]
mae_sarima  = results["SARIMA"]["MAE"]

w_p = (1/mae_prophet) / (1/mae_prophet + 1/mae_sarima)
w_s = (1/mae_sarima)  / (1/mae_prophet + 1/mae_sarima)

pred_ensemble = w_p * results["Prophet"]["pred"] + w_s * results["SARIMA"]["pred"]
mae_ens, rmse_ens = evaluate("앙상블", y_test, pred_ensemble)

print(f"\nProphet 가중치: {w_p:.1%}  |  SARIMA 가중치: {w_s:.1%}")
print(f"앙상블 MAE: {mae_ens:.0f}건  RMSE: {rmse_ens:.0f}건")

# 시각화
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(train["날짜"], y_train, color="steelblue", label="학습", lw=1.5)
ax.plot(test["날짜"], y_test.values, color="black", label="실제", marker="o", lw=2)
ax.plot(test["날짜"], results["Prophet"]["pred"], color="orange", label=f"Prophet (w={w_p:.0%})", ls="--", lw=1.5)
ax.plot(test["날짜"], results["SARIMA"]["pred"],  color="green",  label=f"SARIMA  (w={w_s:.0%})", ls="--", lw=1.5)
ax.plot(test["날짜"], pred_ensemble, color="crimson", label=f"앙상블 MAE={mae_ens:.0f}건", marker="s", lw=2.5)
ax.set_title("Prophet + SARIMA 가중 앙상블 예측")
ax.legend()
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

## 7. 교차 분석 — 공간 위험도 × 시계열 패턴

> 학습 데이터 월별 평균 단속건수로 **고위험 월**을 식별하고,  
> 공간 분석의 **고위험 행정동**과 교차하여 *단속 집중 필요 지역·시기* 도출

In [ ]:
# 시계열: 월별 평균 단속건수 패턴
monthly_avg = train.groupby("월")["단속건수"].mean().round(0)
grand_mean  = monthly_avg.mean()

risk_months = monthly_avg[monthly_avg >= grand_mean].index.tolist()
safe_months = monthly_avg[monthly_avg <  grand_mean].index.tolist()

season_map = {
    1:"겨울",2:"겨울",3:"봄",4:"봄",5:"봄",
    6:"여름",7:"여름",8:"여름",9:"가을",10:"가을",11:"가을",12:"겨울"
}
monthly_df = monthly_avg.reset_index()
monthly_df.columns = ["월","평균단속건수"]
monthly_df["계절"]    = monthly_df["월"].map(season_map)
monthly_df["위험등급"] = monthly_df["평균단속건수"].apply(
    lambda x: "집중단속필요" if x >= grand_mean else "일반관리"
)

print(f"월평균 단속건수 기준: {grand_mean:.0f}건")
print(f"집중 단속 월: {risk_months}")
display(monthly_df)

# 공간: 위험 등급별 행정동
high_dongs = risk[risk["위험등급"] == "고위험"][["행정동명","위험도점수"]]
print("\n고위험 행정동:")
display(high_dongs)

# 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ["crimson" if m in risk_months else "steelblue" for m in monthly_df["월"]]
axes[0].bar(monthly_df["월"], monthly_df["평균단속건수"], color=colors, edgecolor="black")
axes[0].axhline(grand_mean, color="black", ls="--", label=f"기준 {grand_mean:.0f}건")
axes[0].set_xlabel("월")
axes[0].set_ylabel("평균 단속건수")
axes[0].set_title("월별 평균 단속건수\n(빨강=집중 단속 필요 월)")
axes[0].legend()
axes[0].set_xticks(range(1, 13))

top15 = risk.head(15).sort_values("위험도점수")
bar_colors = ["crimson" if g=="고위험" else "orange" if g=="중위험" else "steelblue"
              for g in top15["위험등급"]]
axes[1].barh(top15["행정동명"], top15["위험도점수"], color=bar_colors, edgecolor="black")
axes[1].set_xlabel("위험도 점수")
axes[1].set_title("행정동별 위험도 점수 (상위 15)\n(빨강=고위험, 주황=중위험)")

plt.tight_layout()
plt.savefig("../outputs/cross_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

# 정책 제언 출력
print("=" * 55)
print("  정책 제언: 단속 자원 집중 배분 우선순위")
print("=" * 55)
print(f"집중 단속 필요 월: {risk_months}월")
for _, row in high_dongs.iterrows():
    print(f"최우선 단속 지역: {row['행정동명']}  (위험도 {row['위험도점수']:.2f})")
print("→ 위 지역에 집중 단속 월 기간 중 단속 인력 우선 배치 권고")
print("=" * 55)

## 8. 결과 저장

In [ ]:
comp_df.to_csv("../data/전처리_기존/모델성능비교.csv", encoding="utf-8-sig")
monthly_df.to_csv("../data/전처리_기존/월별단속위험도.csv", index=False, encoding="utf-8-sig")

print("저장 완료")
print("  - data/전처리된 data/모델성능비교.csv")
print("  - data/전처리된 data/월별단속위험도.csv")